<a href="https://colab.research.google.com/github/grmntfrancis0/earthengine-community/blob/main/Landsat_timeseries_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade xee

In [ ]:
!pip install -U geemap

In [ ]:
import ee

In [ ]:
ee.Authenticate()
ee.Initialize(project = "ee-grmntfrancis0",
              opt_url='https://earthengine-highvolume.googleapis.com')

In [ ]:
import geemap

In [ ]:
map = geemap.Map()
map

In [ ]:
roi = map.draw_last_feature.geometry()
roi

In [ ]:
roi = (ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level0")
.filterBounds(roi)
.geometry()
.simplify(100))
map.addLayer(roi, {}, 'roi')

In [ ]:
print(f"Current ROI: {roi.getInfo()}")

In [ ]:
def ndvi_tm_etm(img):
    # Apply scaling factors for Landsat 4-7 Surface Reflectance
    bands = img.multiply(0.0000275).add(-0.2)

    # Calculate NDVI using Near-Infrared (B4) and Red (B3)
    ndvi = bands.normalizedDifference(['SR_B4', 'SR_B3']).rename('ndvi')

    # Return NDVI with original image properties preserved
    return ndvi.copyProperties(img, img.propertyNames())

In [ ]:
landsat4 = (ee.ImageCollection("LANDSAT/LT04/C02/T1_L2")
            .select("SR_B.*")
            .filterDate("1982-01-01", "1994-12-30")
            .filterBounds(roi)
            .filter(ee.Filter.lt("CLOUD_COVER", 45))
            .map(ndvi_tm_etm))

In [ ]:
landsat5 = (ee.ImageCollection("LANDSAT/LT05/C02/T1_L2")
            .select('SR_B.*')
            .filterDate('1984-01-01', '2013-12-30')
            .filterBounds(roi)
            .filter(ee.Filter.lt('CLOUD_COVER', 45))
            .map(ndvi_tm_etm))

In [ ]:
landsat7_slcon = (ee.ImageCollection("LANDSAT/LE07/C02/T1_L2")
                  .select('SR_B.*')
                  .filterDate('1999-01-01', '2003-05-30')
                  .filterBounds(roi)
                  .filter(ee.Filter.lt('CLOUD_COVER', 45))
                  .map(ndvi_tm_etm))

In [ ]:
def slc_off(img):
    # Create a mask where all bands have data (removes the "wedges")
    mask = img.mask().reduce(ee.Reducer.min())

    # Apply the mask and then run the NDVI calculation
    return ndvi_tm_etm(img.updateMask(mask))

In [ ]:
landsat7_slcoff = (ee.ImageCollection("LANDSAT/LE07/C02/T1_L2")
                   .select('SR_B.*')
                   .filterDate('2003-06-01', '2024-12-30')
                   .filterBounds(roi)
                   .filter(ee.Filter.lt('CLOUD_COVER', 45))
                   .map(slc_off))

In [ ]:
map.addLayer(
    landsat7_slcoff.filterDate('2005-01-01', '2006-12-30').toBands().clip(roi), {},
    'slc_off_filled',
    False
)

def ndvi_oli(img):
    bands = img.multiply(0.0000275).add(-0.2)
    ndvi = bands.normalizedDifference(['SR_B5', 'SR_B4']).rename('ndvi')
    return ndvi.copyProperties(img, img.propertyNames())


In [ ]:
landsat8 = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
            .select('SR_B.*')
            .filterDate('2013-01-01', '2024-12-30')
            .filterBounds(roi)
            .filter(ee.Filter.lt('CLOUD_COVER', 45))
            .map(ndvi_oli))

In [ ]:
landsat9 = (ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
            .select('SR_B.*')
            .filterDate('2021-01-01', '2024-12-30')
            .filterBounds(roi)
            .filter(ee.Filter.lt('CLOUD_COVER', 45))
            .map(ndvi_oli))

In [ ]:
landsat_collections = (landsat4.merge(landsat5)
                       .merge(landsat7_slcon)
                       .merge(landsat7_slcoff)
                       .merge(landsat8)
                       .merge(landsat9)
                       .sort('system:time_start'))

In [ ]:
landsat_collections

In [ ]:
import geemap
import geemap.chart as chart

In [ ]:
import geemap.chart as chart

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def get_stats(img):
    mean = img.reduceRegion(reducer=ee.Reducer.mean(), geometry=roi, scale=1000).get('ndvi')
    return img.set('date', img.date().format('YYYY-MM-dd')).set('mean_ndvi', mean)

stats_collection = landsat_collections.map(get_stats)
data = stats_collection.reduceColumns(ee.Reducer.toList(2), ['date', 'mean_ndvi']).get('list').getInfo()

df = pd.DataFrame(data, columns=['date', 'ndvi'])
df['date'] = pd.to_datetime(df['date'])
df['ndvi'] = pd.to_numeric(df['ndvi'], errors='coerce') # Ensure 'ndvi' column is numeric
df.plot(x='date', y='ndvi', title='Long-term NDVI')
plt.show()

In [ ]:
print(f"Number of items in stats_collection: {stats_collection.size().getInfo()}")
print(f"Content of data variable: {data}")